# Near vector search
Run a near vector search for a given query and collection. This is a deeper look into the retrieved contexts than `chatbot_score.ipynb` but harder to read, so it is recommended you start there. A near vector search takes a given query, embeds it using the same model as the collection, and returns the top k results based on similarity score. This is the same logic at the core of rubin_rag, which makes this notebook a great way to test that a collection contains good chunks and embeddings. If this notebook works, then you can rule out the collection being the problem (at least entirely).

In [ ]:
import os
import logging
from dotenv import load_dotenv

from langchain_openai import OpenAIEmbeddings
import weaviate
from weaviate.classes.init import Auth
from weaviate.config import AdditionalConfig, Timeout
from weaviate import WeaviateClient

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
weaviate_api_key = os.getenv("WEAVIATE_API_KEY")
http_host = "weaviate-headless.rubin-rag.svc.cluster.local"
grpc_host = "weaviate-grpc.rubin-rag.svc.cluster.local"

if openai_api_key is None:
    raise ValueError("OPENAI_API_KEY environment variable is not set")
if weaviate_api_key is None:
    raise ValueError("WEAVIATE_API_KEY environment variable is not set")
if http_host is None:
    raise ValueError("HTTP_HOST environment variable is not set")
if grpc_host is None:
    raise ValueError("GRPC_HOST environment variable is not set")

In [ ]:
def near_vector_search(client: WeaviateClient,
                       index: str,
                       query: str,
                       k: int = 5,
                       model: str = "text-embedding-3-large",
                       dimensions: int = 1536,
                      ) -> list[dict]:
    """Run a near vector search on a query. Returns the chunk page content
    and metadata for the best matching chunks best on vector similarity
    score.

    Parameters
    ----------
    client: WeaviateClient
        The connection to Weaviate.
    index: str
        The name of the Weaviate collection to search from.
    query: str
        The question you want to ask the collection.
    k: int, Optional
        k-value, or the number of chunks to retrieve, default is 5.
    model: str
        Text embedding model to use, default is text-embedding-3-large, which
        is what we are using for our chunk embeddings. This must match
        the model used in the collection ingestion.
    dimensions: int, Optional
        Vector dimensions of text embedding model, default is 1536.

    Returns
    -------
    list[dict]
        A list of properties dictionaries for the best matching chunks.
    """
    # Generate the embedding for your query
    embeddings = OpenAIEmbeddings(
        api_key=openai_api_key,
        model=model,
        dimensions=dimensions,
    )
    query_vector = embeddings.embed_query(query)

    # Run the near vector search
    collection = client.collections.get(index)
    response = collection.query.near_vector(
        near_vector=query_vector,
        limit=k
    )

    return [obj.properties for obj in response.objects]

In [ ]:
def pretty_print(results: list[dict], skip_none: bool = True):
    """Pretty printing function for results of near vector search.
    Includes an toggle to ignore metadata that have None values.
    """
    for i, result in enumerate(results, 1):
        print(f"\n=============== Result {i} ===============")
        for key, value in result.items():
            if skip_none and value is None:
                continue
            print(f"{key}:")
            if isinstance(value, str) and len(value) > 100:
                lines = value.strip().split("\n")
                for line in lines:
                    print(f"    {line}")
            else:
                print(f"    {value}")

In [ ]:
try:
    client = weaviate.connect_to_custom(
        http_host=http_host,
        http_port=8080,
        http_secure=False,
        grpc_host=grpc_host,
        grpc_port=50051,
        grpc_secure=False,
        auth_credentials=Auth.api_key(
            weaviate_api_key
        ),
        headers={"X-OpenAI-Api-Key": openai_api_key},
        additional_config=AdditionalConfig(
            timeout=Timeout(init=30, query=60, insert=120)
        )
    )

    results = near_vector_search(client=client,
                                 index="Ingestion_20250610",
                                 query="how to setup the notebooks_pingraham repo?",
                                 k=5,
                                 model="text-embedding-3-small",
                                 dimensions=1536
                                )
    
    pretty_print(results)

except Exception as e:
    print(e)
finally:
    client.close()